# MOSAIC Reference Code Set Detector v1
**Detection-only.** Read-only against source data. Every finding requires steward review before any action.

## What this detects

Reference code sets are the most complex domain in MOSAIC DQ because each standard has its own valid value set, companion column requirements, rollup hierarchy rules, and PHI sensitivity level. This detector evaluates:

| Code domain | Governing standard | Key checks |
|---|---|---|
| Administrative gender | HL7 FHIR AdministrativeGender / ISO 5218 | Values in {M,F,O,U,N}; no conflation; companion system |
| Sex/gender fields (clinical) | USCDI SOGI | Separate columns per concept; PHI sensitivity |
| Race | OMB SPD 15 + CDC CDCREC | OMB category validity; CDCREC code system; rollup lookup; version tracking |
| Ethnicity | OMB SPD 15 + CDC CDCREC | Same as race; 1997 vs 2024 transition tracking |
| Country | ISO 3166-1 alpha-2 | Alpha-2 canonical; no alpha-3 or numeric in certified dims |
| Language | ISO 639 + BCP 47 | BCP 47 tag format; companion system column |
| Currency | ISO 4217 | Three-letter uppercase codes |
| Units of measure | UCUM | Clinical lab/vitals values |
| Clinical terminologies | ICD-10-CM, LOINC, SNOMED CT, RxNorm, CVX, NDC, NPI | Code + system + version; no free-text |

## Column group detection

The detector identifies **code groups** per table: for each coded column it finds companion `_code_system`, `_std_version`, and `_version` columns in the same table. Findings are reported at both the code column level and the group (table) level.

| Cell | What it does |
|---|---|
| 1-2 | Overview and Rule Reference |
| 3 | Config / Widgets |
| 4 | Constants: code sets, column patterns, standardization rules |
| 5 | Discovery: identify coded columns and companion system/version columns |
| 6 | Profiler: one SQL per table; distinct values, null rates, format distribution |
| 7 | AI Classifier: confirm code system, detect free-text vs code, detect conflation |
| 8 | Rule Engine: evaluate each code group against its governing standard |
| 9 | Write Results |
| 10 | Summary: blocking → by standard → by table → PHI risk → work queue |

> **Hard constraint:** UPDATE, MERGE, DELETE, CTAS on source tables are never executed.


# MOSAIC Reference Code Set Detector — Rule Reference

**Detection-only.** No source data is modified.

---

## §Standards catalog (§1, §6)
| Rule tag | What it detects | Required action |
|---|---|---|
| `MISSING_CODE_SYSTEM` | Coded column has no companion `_code_system` column in the same table | Add `_code_system` column; document default system in data dictionary |
| `MISSING_VERSION` | Certified coded column has no `_std_version` or `_version` companion column | Add version tracking; record which standard version values were collected under |
| `WRONG_CODE_SYSTEM` | The `_code_system` column value doesn't match the expected standard for this code type | Correct the code system designation; verify source mapping |
| `FREE_TEXT_WHERE_CODE_EXISTS` | Column stores free-text descriptions (e.g. "Male", "White") for an attribute with a governing code standard | Replace with coded values in Silver ETL; flag certified columns |

## §General handling (§2)
| Rule tag | What it detects | Required action |
|---|---|---|
| `OUT_OF_SET_VALUE` | Code value not in the governing code set for its declared system | NULL or quarantine; do not pass to Gold without mapping |
| `HIGH_OOS_RATE` | > threshold % of non-null values are out-of-set | Escalate to steward; certification blocker |
| `BARE_LABEL_STORED` | Storing full labels (Male/Female, White/Black) where short codes are the standard | Map to canonical codes in Silver ETL; register in alias table |
| `VERSION_NOT_TRACKED` | Coded column present with no version column and no documented default | Add version tracking; critical for race/ethnicity OMB 1997 vs 2024 transition |

## §Sex and gender (§3)
| Rule tag | What it detects | Required action |
|---|---|---|
| `INVALID_ADMIN_GENDER` | Administrative gender value not in {M, F, O, U, N} | Map to canonical codes; unmapped → U or quarantine per steward rule |
| `GENDER_FIELDS_CONFLATED` | Single column attempting to store multiple gender concepts (birth sex + gender identity combined) | Separate into distinct columns per USCDI; document each field's concept |
| `SOGI_PHI_RISK` | Gender identity or sexual orientation column detected | Verify PHI access controls; minimum-necessary access enforcement |
| `MISSING_BIRTH_SEX` | Table appears to track patient demographics but has gender but no birth_sex column | Add sex_assigned_at_birth / birth_sex column per USCDI |

## §Race and ethnicity (§4)
| Rule tag | What it detects | Required action |
|---|---|---|
| `INVALID_OMB_CATEGORY` | Race or ethnicity value not in OMB minimum categories (1997 or 2024) | Map to valid OMB categories via CDCREC rollup |
| `VERSION_TRANSITION_RISK` | Race/ethnicity column present without OMB version tracking (1997 vs 2024) | Add `race_std_version` column; critical for future OMB 2024 compliance |
| `MISSING_ROLLUP_LOOKUP` | Detailed CDCREC codes present but no rollup hierarchy table detected | Create/register rollup lookup per §4; do not discard source granularity |
| `MENA_NOT_MODELED` | Column uses OMB 1997 categories only with no MENA provision for OMB 2024 transition | Plan forward-compatible model per §4 |

## §Validation gates (§6)
| Rule tag | What it detects | Required action |
|---|---|---|
| `MISSING_CODE_SYSTEM` | (also listed above) — certification gate | Add system column before Gold promotion |
| `UNCODED_FREE_TEXT_CERTIFIED` | Free-text in a certified/Gold column for a standardized attribute | Certification blocker; replace with coded values |

---

## Enforcement
- Certified demographic/coded dimensions with non-standard codes, missing code system/version, or uncoded free-text where a standard exists are **certification blockers**.
- Discarding source granularity (no rollup lookup) or retro-mapping across standard versions without steward sign-off is a **policy violation**.


In [0]:
import re, json, uuid
from datetime import datetime
from typing import Any, Dict, List, Tuple, Set, Optional
from collections import defaultdict
from pyspark.sql import functions as F, DataFrame

def _w(name: str, default) -> str:
    try:
        dbutils.widgets.text(name, str(default))
        return dbutils.widgets.get(name)
    except Exception:
        return str(default)

CFG: Dict[str, Any] = {
    "catalog":      _w("catalog",      "data_classification_source"),
    "schema":       _w("schema",       "reference_code_sets_detector"),
    "table_filter": _w("table_filter", ""),
    "skip_views":   _w("skip_views",   "true").strip().lower() == "true",
    "layer":        _w("layer",        "Unknown"),
    "sample_size":  int(_w("sample_size",  10_000)),
    "seed":         int(_w("seed",         42)),
    # % non-null out-of-set before HIGH_OOS_RATE fires
    "oos_threshold":    float(_w("oos_threshold",    "2.0")),
    # % null before HIGH_NULL_RATE fires
    "null_threshold":   float(_w("null_threshold",   "10.0")),
    # % of values that are free-text labels (not codes) before FREE_TEXT_WHERE_CODE_EXISTS fires
    "label_threshold":  float(_w("label_threshold",  "5.0")),
    # AI
    "ai_model":  _w("ai_model",  "databricks-meta-llama-3-3-70b-instruct"),
    "enable_ai": _w("enable_ai", "true").strip().lower() == "true",
    # Results
    "out_catalog": _w("out_catalog", "data_classification_target"),
    "out_schema":  _w("out_schema",  "data_classification_output"),
    "out_table":   _w("out_table",   "refcode_findings"),
}

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.utcnow().isoformat()

print(f"Run     : {RUN_ID}")
print(f"Scope   : {CFG['catalog']}.{CFG['schema']}")
print(f"Layer   : {CFG['layer']}")
print(f"AI      : {'on -> ' + CFG['ai_model'] if CFG['enable_ai'] else 'off'}")
print(f"Results : {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")


In [0]:
# ---------------------------------------------------------------------------
# CONSTANTS — code sets, column name heuristics, standardization rules
# ---------------------------------------------------------------------------

TAGS = {
    "MISSING_CODE_SYSTEM":          "§Standards-catalog",
    "MISSING_VERSION":              "§Standards-catalog",
    "WRONG_CODE_SYSTEM":            "§Standards-catalog",
    "FREE_TEXT_WHERE_CODE_EXISTS":  "§Standards-catalog",
    "OUT_OF_SET_VALUE":             "§Validation",
    "HIGH_OOS_RATE":                "§Validation",
    "HIGH_NULL_RATE":               "§Validation",
    "BARE_LABEL_STORED":            "§General-handling",
    "VERSION_NOT_TRACKED":          "§General-handling",
    "INVALID_ADMIN_GENDER":         "§Sex-gender",
    "GENDER_FIELDS_CONFLATED":      "§Sex-gender",
    "SOGI_PHI_RISK":                "§Sex-gender",
    "MISSING_BIRTH_SEX":            "§Sex-gender",
    "INVALID_OMB_CATEGORY":         "§Race-ethnicity",
    "VERSION_TRANSITION_RISK":      "§Race-ethnicity",
    "MISSING_ROLLUP_LOOKUP":        "§Race-ethnicity",
    "MENA_NOT_MODELED":             "§Race-ethnicity",
    "UNCODED_FREE_TEXT_CERTIFIED":  "§Validation",
}

SEVERITY: Dict[str, int] = {
    "UNCODED_FREE_TEXT_CERTIFIED":   10,
    "OUT_OF_SET_VALUE":              10,
    "HIGH_OOS_RATE":                  9,
    "INVALID_ADMIN_GENDER":           9,
    "INVALID_OMB_CATEGORY":           9,
    "MISSING_CODE_SYSTEM":            8,
    "FREE_TEXT_WHERE_CODE_EXISTS":    8,
    "GENDER_FIELDS_CONFLATED":        8,
    "MISSING_ROLLUP_LOOKUP":          8,
    "BARE_LABEL_STORED":              7,
    "VERSION_TRANSITION_RISK":        7,
    "WRONG_CODE_SYSTEM":              7,
    "MISSING_VERSION":                6,
    "VERSION_NOT_TRACKED":            6,
    "SOGI_PHI_RISK":                  6,
    "MISSING_BIRTH_SEX":              5,
    "MENA_NOT_MODELED":               4,
    "HIGH_NULL_RATE":                 4,
}

# ── Code domain taxonomy ─────────────────────────────────────────────────────
CODE_DOMAINS = {
    "admin_gender",    # FHIR AdministrativeGender / ISO 5218
    "birth_sex",       # USCDI sex assigned at birth
    "gender_identity", # USCDI SOGI -- PHI
    "sexual_orientation", # USCDI SOGI -- PHI
    "race",            # OMB SPD 15 + CDCREC
    "ethnicity",       # OMB SPD 15 + CDCREC
    "country",         # ISO 3166-1 alpha-2
    "subdivision",     # ISO 3166-2
    "language",        # ISO 639 / BCP 47
    "currency",        # ISO 4217
    "ucum_unit",       # UCUM
    "icd",             # ICD-10-CM diagnosis
    "loinc",           # LOINC lab/obs
    "snomed",          # SNOMED CT
    "rxnorm",          # RxNorm
    "cvx",             # CVX immunization
    "ndc",             # NDC drug
    "npi",             # NPI provider
    "other_code",      # coded but domain unclear
    "not_a_code",      # AI determined this is not a reference code column
}

# ── Column name heuristics ───────────────────────────────────────────────────
RE_GENDER = re.compile(
    r"\b(gender|sex|administrative_gender|admin_gender|"
    r"gender_code|sex_code|gender_identity|gender_id|"
    r"birth_sex|sex_at_birth|assigned_sex|sexual_orientation|sogi)\b",
    re.I
)
RE_RACE = re.compile(
    r"\b(race|race_code|race_cd|race_cat|racial|"
    r"race_omb|race_cdcrec|race_category|race_group)\b",
    re.I
)
RE_ETHNICITY = re.compile(
    r"\b(ethnicity|ethnic|ethnicity_code|ethnic_code|"
    r"ethnicity_cat|hispanic|hispanic_indicator|hispanic_flag)\b",
    re.I
)
RE_COUNTRY = re.compile(
    r"\b(country|country_code|country_cd|iso_country|"
    r"country_of_origin|birth_country|nation|nationality)\b",
    re.I
)
RE_LANGUAGE = re.compile(
    r"\b(language|language_code|lang_code|preferred_language|"
    r"primary_language|spoken_language|written_language|"
    r"interpreter_language|bcp47|iso639)\b",
    re.I
)
RE_CURRENCY = re.compile(
    r"\b(currency|currency_code|currency_cd|iso4217|"
    r"payment_currency|transaction_currency)\b",
    re.I
)
RE_UCUM = re.compile(
    r"\b(unit|units|unit_of_measure|uom|ucum|"
    r"result_unit|value_unit|measurement_unit|"
    r"lab_unit|obs_unit|vital_unit)\b",
    re.I
)
RE_CLINICAL = re.compile(
    r"\b(icd|icd10|icd_10|icd9|diagnosis_code|diag_code|dx_code|"
    r"loinc|loinc_code|obs_code|lab_code|"
    r"snomed|sct_code|clinical_code|"
    r"rxnorm|rx_norm|rxcui|drug_code|medication_code|"
    r"cvx|vaccine_code|immunization_code|"
    r"ndc|ndc_code|drug_ndc|"
    r"npi|provider_npi|npi_code|"
    r"procedure_code|proc_code|cpt|hcpcs)\b",
    re.I
)
RE_CODE_SYSTEM = re.compile(
    r"\b(code_system|coding_system|code_system_id|"
    r"terminology|vocab|vocabulary|ontology|"
    r"code_source|standard_name|standard_id)\b",
    re.I
)
RE_VERSION = re.compile(
    r"\b(std_version|standard_version|code_version|"
    r"version|spec_version|omb_version|cdcrec_version|"
    r"terminology_version|vocab_version)\b",
    re.I
)
RE_ROLLUP = re.compile(
    r"\b(rollup|roll_up|omb_category|omb_min|minimum_category|"
    r"race_rollup|ethnicity_rollup|parent_code|category_code)\b",
    re.I
)

# ── Canonical code sets ───────────────────────────────────────────────────────

# FHIR AdministrativeGender canonical (§3)
ADMIN_GENDER_CODES = {"M", "F", "O", "U", "N"}
# Common label variants that should be mapped to codes (detect BARE_LABEL)
ADMIN_GENDER_LABELS = {
    "MALE", "FEMALE", "OTHER", "UNKNOWN", "NOT APPLICABLE",
    "NOT KNOWN", "MALE GENDER", "FEMALE GENDER",
}

# ISO 5218 numeric codes (informational -- detect if stored instead of FHIR)
ISO5218_CODES = {"0", "1", "2", "9"}

# SOGI columns -- PHI sensitive
SOGI_NAMES = re.compile(
    r"\b(gender_identity|sexual_orientation|sogi|"
    r"gender_id|sexual_orient)\b", re.I
)

# OMB 1997 race minimum categories (CDCREC codes)
OMB_RACE_1997_CODES = {
    "1002-5",  # American Indian or Alaska Native
    "2028-9",  # Asian
    "2054-5",  # Black or African American
    "2076-8",  # Native Hawaiian or Other Pacific Islander
    "2106-3",  # White
    "2131-1",  # Some Other Race
    "AIAN", "ASIAN", "BLACK", "NHOPI", "WHITE", "OTHER",  # common display codes
    "A", "B", "W", "I", "P",  # abbreviated forms
}
OMB_RACE_1997_LABELS = {
    "AMERICAN INDIAN OR ALASKA NATIVE", "ASIAN",
    "BLACK OR AFRICAN AMERICAN", "NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER",
    "WHITE", "SOME OTHER RACE", "MULTIRACIAL", "TWO OR MORE RACES",
}
# OMB 2024 adds MENA
OMB_RACE_2024_ADDITIONS = {
    "MENA", "MIDDLE EASTERN OR NORTH AFRICAN", "2190-7",
}

# OMB 1997 ethnicity minimum categories
OMB_ETHNICITY_1997_CODES = {
    "2135-2",  # Hispanic or Latino
    "2186-5",  # Not Hispanic or Latino
    "H", "NH",  # abbreviated
    "HISPANIC", "NON-HISPANIC", "NOT HISPANIC",
}
OMB_ETHNICITY_LABELS = {
    "HISPANIC OR LATINO", "NOT HISPANIC OR LATINO",
    "HISPANIC", "NON-HISPANIC", "DECLINED", "UNKNOWN",
}

# ISO 3166-1 alpha-2 — most common codes (full list would be 250+)
ISO3166_ALPHA2 = {
    "US","CA","MX","GB","FR","DE","JP","CN","AU","BR","IN","KR","IT","ES","NL",
    "SE","NO","DK","FI","CH","AT","BE","PT","PL","CZ","HU","RO","GR","TR","IL",
    "ZA","NG","EG","KE","MA","GH","ET","TZ","UG","TH","VN","PH","ID","MY","SG",
    "NZ","AR","CL","CO","PE","VE","EC","BO","PY","UY","PR","DO","CU","HT","GT",
    "HN","SV","NI","CR","PA","JM","TT","BB","LC","VC","GD","AG","DM","KN","BS",
    "TC","VG","KY","AI","BM","MS","AW","CW","SX","BQ","MF","GP","MQ","GF","PM",
    "UM","AS","GU","MP","VI","PR","AE","SA","QA","KW","BH","OM","YE","IQ","IR",
    "AF","PK","BD","LK","NP","MM","KH","LA","BN","TL","MN","KZ","UZ","TM","KG",
    "TJ","AZ","AM","GE","BY","UA","MD","RS","HR","BA","SI","MK","AL","ME","XK",
    "BG","LV","LT","EE","FI","IS","LI","LU","MT","CY","SK","AD","MC","SM","VA",
    "RU","EE","LV","LT",
}
# ISO 3166-1 alpha-3 — should NOT be stored as canonical (detect and flag)
ISO3166_ALPHA3_SAMPLE = {"USA","CAN","MEX","GBR","FRA","DEU","JPN","CHN","AUS","BRA"}

# ISO 639 / BCP 47 common language codes
BCP47_COMMON = {
    "en","en-US","en-GB","es","es-US","es-MX","zh","zh-CN","zh-TW",
    "fr","fr-FR","de","de-DE","pt","pt-BR","pt-PT","ar","ar-SA",
    "ko","ko-KR","ja","ja-JP","vi","vi-VN","tl","tl-PH",
    "ru","ru-RU","pl","pl-PL","it","it-IT","nl","nl-NL",
    "hi","hi-IN","ur","ur-PK","bn","bn-BD","gu","pa","ta","te",
    "hmn","hy","ka","uk","ro","hu","cs","sk","hr","sr","bg",
    "he","he-IL","fa","fa-IR","tr","tr-TR","sw","am","so","km","ky",
}

# ISO 4217 currency codes
ISO4217_CODES = {
    "USD","EUR","GBP","CAD","AUD","JPY","CHF","CNY","SEK","NOK","DKK",
    "NZD","MXN","BRL","INR","KRW","SGD","HKD","TWD","ZAR","RUB","TRY",
    "AED","SAR","QAR","KWD","BHD","ILS","EGP","NGN","KES","GHS","MAD",
    "COP","PEN","ARS","CLP","PYG","BOB","UYU","DOP","CUP","GTQ","HNL",
    "SVC","NIO","CRC","PAB","JMD","TTD","BBD","BSD","KYD","BMD","AWG",
}

# UCUM common clinical units
UCUM_COMMON = {
    "mg/dL","mmol/L","g/dL","mg/L","ug/dL","ng/mL","pg/mL","IU/L","U/L",
    "mmHg","cm H2O","kPa","mbar",
    "kg","g","lb","oz","mg","ug",
    "cm","m","in","ft","mm",
    "mL","L","dL","uL","fl oz",
    "cel","[degF]",  # temperature
    "min","h","d","wk","mo","a",  # time
    "{cells}/uL","{beats}/min","breaths/min",  # counts
    "%","10*3/uL","10*6/uL","10*9/L",  # ratios/concentrations
    "mEq/L","mmol/mol","umol/L",
}

# Clinical code system names (expected values in _code_system columns)
CLINICAL_SYSTEMS_EXPECTED = {
    "icd":    {"ICD-10-CM", "ICD-10-PCS", "ICD-9-CM", "ICD-10"},
    "loinc":  {"LOINC"},
    "snomed": {"SNOMED CT", "SCT", "SNOMEDCT"},
    "rxnorm": {"RxNorm", "RXNORM"},
    "cvx":    {"CVX"},
    "ndc":    {"NDC"},
    "npi":    {"NPI"},
    "race":   {"CDCREC", "HL7", "OMB", "US-CORE-RACE"},
    "ethnicity": {"CDCREC", "HL7", "OMB", "US-CORE-ETHNICITY"},
    "country": {"ISO3166-1", "ISO 3166-1", "ISO3166"},
    "language": {"ISO639", "ISO 639", "BCP47", "BCP 47", "IETF"},
    "currency": {"ISO4217", "ISO 4217"},
}

STANDARDIZATION_RULES: Dict[str, list] = {

    "MISSING_CODE_SYSTEM": [
        {"option_key": "add_code_system_column",
         "label":      "Add companion _code_system column with documented default",
         "sql":        "-- ALTER TABLE ... ADD COLUMN <col>_code_system STRING DEFAULT '<system>'",
         "notes":      "Store code + code system + version per §2. "
                       "Never store a bare label without its system. "
                       "Document default system in data dictionary when all values share one system."},
    ],

    "MISSING_VERSION": [
        {"option_key": "add_version_column",
         "label":      "Add companion _std_version column with documented default",
         "sql":        "-- ALTER TABLE ... ADD COLUMN <col>_std_version STRING DEFAULT '<version>'",
         "notes":      "Version transitions must be tracked, not overwritten (§2). "
                       "Critical for race/ethnicity: OMB 1997 vs OMB 2024 transition."},
    ],

    "WRONG_CODE_SYSTEM": [
        {"option_key": "correct_code_system",
         "label":      "Correct the code_system value to match the governing standard",
         "sql":        "-- UPDATE <col>_code_system SET value = '<correct_system>' "
                       "WHERE <col>_code_system = '<wrong_system>'",
         "notes":      "Verify source mapping and correct the system designation. "
                       "Certified coded columns must carry their correct code system (§6)."},
    ],

    "FREE_TEXT_WHERE_CODE_EXISTS": [
        {"option_key": "map_to_codes",
         "label":      "Replace free-text with coded values in Silver ETL",
         "sql":        "-- SILVER: JOIN alias_table ON UPPER(TRIM(col)) = alias_table.source_value "
                       "→ emit alias_table.canonical_code AS col",
         "notes":      "No free-text where a standard exists (§6). "
                       "Certified columns storing uncoded text for standardized attributes "
                       "are certification blockers (§Enforcement)."},
    ],

    "OUT_OF_SET_VALUE": [
        {"option_key": "null_oos",
         "label":      "Set out-of-set values to NULL or U (unknown) in Silver",
         "sql":        "-- SILVER: CASE WHEN UPPER(TRIM(col)) IN (<canonical_set>) "
                       "THEN UPPER(TRIM(col)) ELSE NULL END AS col",
         "notes":      "Out-of-set values fail or quarantine per §6 validation gate. "
                       "For administrative gender: unmapped → U or quarantine per steward rule."},
        {"option_key": "quarantine_oos",
         "label":      "Route out-of-set rows to quarantine",
         "sql":        "-- Route rows WHERE UPPER(TRIM(col)) NOT IN (<canonical_set>) to quarantine.",
         "notes":      "Do not pass non-standard codes to Gold/certified dimensions."},
    ],

    "HIGH_OOS_RATE": [
        {"option_key": "escalate_and_quarantine",
         "label":      "Escalate to steward; quarantine all out-of-set rows",
         "sql":        "-- Route all OOS rows to quarantine. Block certification.",
         "notes":      "Certified demographic/coded dimensions with non-standard codes "
                       "are certification blockers (§Enforcement)."},
    ],

    "HIGH_NULL_RATE": [
        {"option_key": "investigate_source",
         "label":      "Investigate source for upstream mapping failure or data loss",
         "sql":        "-- No transform. Investigate source system.",
         "notes":      "High null rate in coded column may indicate upstream mapping failure."},
    ],

    "BARE_LABEL_STORED": [
        {"option_key": "map_to_codes",
         "label":      "Map full labels to canonical codes in Silver ETL",
         "sql":        "-- SILVER: CASE WHEN UPPER(TRIM(col)) = 'MALE' THEN 'M' "
                       "WHEN UPPER(TRIM(col)) = 'FEMALE' THEN 'F' ... END AS col",
         "notes":      "Store canonical codes, not display labels (§2, §3). "
                       "Register label → code mappings in alias table. "
                       "Never both label and code in one column without a conversion layer."},
    ],

    "VERSION_NOT_TRACKED": [
        {"option_key": "add_version_tracking",
         "label":      "Add version tracking column and record collection version",
         "sql":        "-- ADD COLUMN <col>_std_version STRING DEFAULT 'OMB-1997'",
         "notes":      "Version transitions tracked, not overwritten (§2). "
                       "For race/ethnicity: OMB 2024 finalized 2024-03-28; "
                       "federal compliance by 2029-03-28. Model must be additive."},
    ],

    "INVALID_ADMIN_GENDER": [
        {"option_key": "map_to_canonical",
         "label":      "Map to canonical FHIR AdministrativeGender codes {M,F,O,U,N}",
         "sql":        "-- SILVER: CASE WHEN UPPER(TRIM(col)) IN ('M','MALE') THEN 'M' "
                       "WHEN UPPER(TRIM(col)) IN ('F','FEMALE') THEN 'F' "
                       "WHEN UPPER(TRIM(col)) IN ('O','OTHER') THEN 'O' "
                       "WHEN UPPER(TRIM(col)) IN ('N','NOT APPLICABLE') THEN 'N' "
                       "ELSE 'U' END AS col",
         "notes":      "Post-transform administrative gender must be in {M,F,O,U,N} (§3). "
                       "Unmapped values → U or quarantine per steward rule -- document which."},
    ],

    "GENDER_FIELDS_CONFLATED": [
        {"option_key": "separate_gender_fields",
         "label":      "Separate into distinct columns per USCDI concept",
         "sql":        "-- SILVER: admin_gender, birth_sex, gender_identity as separate columns.",
         "notes":      "Do not conflate sex/gender concepts into one column (§3). "
                       "Where source collects separately, store separately and document."},
    ],

    "SOGI_PHI_RISK": [
        {"option_key": "verify_phi_controls",
         "label":      "Verify PHI access controls and minimum-necessary access",
         "sql":        "-- No transform. Verify column-level security in Unity Catalog.",
         "notes":      "Gender identity and sexual orientation are PHI-sensitive (§3). "
                       "Apply PHI storage and handling; minimum-necessary access enforcement."},
    ],

    "MISSING_BIRTH_SEX": [
        {"option_key": "add_birth_sex_column",
         "label":      "Add sex_assigned_at_birth / birth_sex column per USCDI",
         "sql":        "-- ADD COLUMN birth_sex STRING -- distinct from admin gender",
         "notes":      "USCDI requires sex assigned at birth as distinct from gender identity (§3). "
                       "Do not conflate with administrative gender column."},
    ],

    "INVALID_OMB_CATEGORY": [
        {"option_key": "map_to_omb_category",
         "label":      "Map detailed CDCREC code to OMB minimum category via rollup lookup",
         "sql":        "-- SILVER: JOIN cdcrec_rollup ON source_code = cdcrec_rollup.detailed_code "
                       "→ emit cdcrec_rollup.omb_category",
         "notes":      "Keep most granular source value; roll up via CDCREC hierarchy (§4). "
                       "Do not discard granularity at ingest. "
                       "Store OMB minimum category + detailed CDCREC code + version."},
    ],

    "VERSION_TRANSITION_RISK": [
        {"option_key": "add_omb_version_tracking",
         "label":      "Add race_std_version column; track OMB 1997 vs 2024",
         "sql":        "-- ADD COLUMN race_std_version STRING DEFAULT 'OMB-1997'",
         "notes":      "OMB 2024 finalized 2024-03-28; federal compliance by 2029-03-28 (§4). "
                       "Model must be additive for future MENA category and combined question. "
                       "Do not retro-map historical data without steward sign-off."},
    ],

    "MISSING_ROLLUP_LOOKUP": [
        {"option_key": "create_rollup_lookup",
         "label":      "Create CDCREC → OMB rollup hierarchy lookup table",
         "sql":        "-- CREATE TABLE cdcrec_rollup ("
                       "detailed_code STRING, omb_category STRING, "
                       "omb_version STRING, effective_from DATE)",
         "notes":      "Every detailed CDCREC value maps to exactly one OMB minimum category (§6). "
                       "Discarding source granularity without rollup is a policy violation (§Enforcement)."},
    ],

    "MENA_NOT_MODELED": [
        {"option_key": "plan_mena_addition",
         "label":      "Plan additive model change for OMB 2024 MENA category",
         "sql":        "-- No immediate transform. Plan schema extension for MENA category.",
         "notes":      "OMB 2024 adds MENA (Middle Eastern or North African) as a 7th minimum category. "
                       "Federal compliance required by 2029-03-28. "
                       "Model changes must be additive -- do not hard-code 5+2 shape."},
    ],

    "UNCODED_FREE_TEXT_CERTIFIED": [
        {"option_key": "replace_with_coded_values",
         "label":      "Replace free-text with coded values before Gold promotion",
         "sql":        "-- Block Gold promotion until coded values are in place.",
         "notes":      "Certification blocker: certified columns must not store uncoded free-text "
                       "for standardized attributes (§6, §Enforcement)."},
    ],
}

print(f"Constants loaded: {len(TAGS)} tags | {len(STANDARDIZATION_RULES)} standardization entries")
print(f"Admin gender codes: {sorted(ADMIN_GENDER_CODES)}")
print(f"OMB race 1997: {len(OMB_RACE_1997_CODES)} codes")
print(f"ISO 3166 alpha-2: {len(ISO3166_ALPHA2)} codes")
print(f"BCP47 common: {len(BCP47_COMMON)} codes")


In [0]:
# ---------------------------------------------------------------------------
# DISCOVERY:
# For each table, identify ALL reference-code-related columns and their
# companion code_system and version columns.
# Builds: table_code_groups -- per table, per column: domain, companion cols,
#         data type, and whether it is a code_system / version column itself.
# ---------------------------------------------------------------------------

cat, sch = CFG["catalog"], CFG["schema"]

def _esc(name: str) -> str:
    return name.replace("`", "``")

view_clause = "AND table_type IN ('MANAGED', 'EXTERNAL')" if CFG["skip_views"] else ""
tables: List[str] = [
    r.table_name for r in spark.sql(f"""
        SELECT table_name FROM `{_esc(cat)}`.information_schema.tables
        WHERE  table_schema = '{_esc(sch)}' {view_clause}
        ORDER BY table_name
    """).collect()
]

if CFG["table_filter"].strip():
    pat = re.compile(CFG["table_filter"], re.I)
    tables = [t for t in tables if pat.search(t)]

if not tables:
    raise ValueError(f"No tables found in {cat}.{sch}")

tbl_in = ", ".join(f"'{_esc(t)}'" for t in tables)

# Collect all columns per table (name + dtype)
table_all_cols: Dict[str, List[Tuple[str, str]]] = defaultdict(list)
for r in spark.sql(f"""
    SELECT table_name, column_name, data_type
    FROM `{_esc(cat)}`.information_schema.columns
    WHERE table_schema = '{_esc(sch)}' AND table_name IN ({tbl_in})
    ORDER BY table_name, ordinal_position
""").collect():
    table_all_cols[r.table_name].append((r.column_name, r.data_type.upper()))


def _heuristic_domain(col: str) -> str:
    # Name-based heuristic domain classification.
    if SOGI_NAMES.search(col):
        if re.search(r"gender_identity|gender_id", col, re.I): return "gender_identity"
        if re.search(r"sexual_orient|sogi", col, re.I):       return "sexual_orientation"
    if RE_GENDER.search(col):
        if re.search(r"birth_sex|sex_at_birth|assigned_sex", col, re.I): return "birth_sex"
        return "admin_gender"
    if RE_RACE.search(col):      return "race"
    if RE_ETHNICITY.search(col): return "ethnicity"
    if RE_COUNTRY.search(col):   return "country"
    if RE_LANGUAGE.search(col):  return "language"
    if RE_CURRENCY.search(col):  return "currency"
    if RE_UCUM.search(col):      return "ucum_unit"
    if RE_CLINICAL.search(col):
        cl = col.lower()
        if "icd" in cl or "diag" in cl or "dx" in cl:  return "icd"
        if "loinc" in cl or "obs_code" in cl or "lab_code" in cl: return "loinc"
        if "snomed" in cl or "sct" in cl:              return "snomed"
        if "rxnorm" in cl or "rxcui" in cl or "drug_code" in cl: return "rxnorm"
        if "cvx" in cl or "vaccine" in cl or "immun" in cl: return "cvx"
        if "ndc" in cl:                                 return "ndc"
        if "npi" in cl:                                 return "npi"
        return "other_code"
    return "not_a_code"


# Build code groups per table
# For each code column, find companion _code_system and _version columns
# by checking if a column name starts with the same prefix

table_code_groups: Dict[str, Dict[str, dict]] = {}

for tbl, cols in table_all_cols.items():
    col_names    = {c.lower(): c for c, _ in cols}  # lower → original
    col_dtypes   = {c.lower(): dt for c, dt in cols}
    code_cols    = {}  # original_col → info

    for col, dtype in cols:
        domain = _heuristic_domain(col)
        if domain == "not_a_code":
            continue
        # Skip columns that ARE the code_system or version companions
        if RE_CODE_SYSTEM.search(col) or RE_VERSION.search(col) or RE_ROLLUP.search(col):
            continue

        # Find companion code_system column:
        # look for <col>_code_system, <col>_system, <prefix>_code_system
        col_lower  = col.lower()
        companion_system  = None
        companion_version = None
        companion_rollup  = None

        # Search all columns for system/version companions of this column
        for other_col_lower, other_orig in col_names.items():
            if other_col_lower == col_lower:
                continue
            if RE_CODE_SYSTEM.search(other_col_lower):
                # Accept if the other col starts with same base as this col
                # e.g. race_code → race_code_system, race → race_code_system
                base = col_lower.replace("_code","").replace("_cd","")
                if other_col_lower.startswith(base) or col_lower.replace("_code","") in other_col_lower:
                    companion_system = other_orig
            if RE_VERSION.search(other_col_lower):
                base = col_lower.replace("_code","").replace("_cd","")
                if other_col_lower.startswith(base) or col_lower.replace("_code","") in other_col_lower:
                    companion_version = other_orig
            if RE_ROLLUP.search(other_col_lower):
                companion_rollup = other_orig

        code_cols[col] = {
            "dtype":              dtype,
            "domain":             domain,
            "heuristic_domain":   domain,
            "companion_system":   companion_system,
            "companion_version":  companion_version,
            "companion_rollup":   companion_rollup,
            # Profiler stats (filled in Cell 4)
            "total":              0,
            "null_count":         0,
            "distinct_vals":      {},
            "sample_vals":        [],
            "oos_count":          0,
            "label_count":        0,
            # AI results (filled in Cell 5)
            "ai_domain":          domain,
            "ai_confirmed":       True,
            "ai_confidence":      "low",
            "is_free_text":       False,
        }

    if code_cols:
        table_code_groups[tbl] = code_cols

print(f"Scope       : {cat}.{sch}")
print(f"Tables      : {len(tables)}")
print(f"Code tables : {len(table_code_groups)}")
for tbl, cols in table_code_groups.items():
    domains = [info['domain'] for info in cols.values()]
    print(f"  {tbl}: {list(cols.keys())} → {domains}")


In [0]:
# ---------------------------------------------------------------------------
# PROFILER -- one SQL per table for ALL code columns simultaneously.
# Per column: null_count, total, distinct values with counts,
#             out-of-set count (domain-specific), label count (free-text check).
# Domain-specific checks are computed in-SQL using CASE expressions.
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")

def get_sample(tbl: str) -> DataFrame:
    fq = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    n  = CFG["sample_size"]
    try:
        return spark.sql(f"SELECT * FROM {fq} TABLESAMPLE ({n} ROWS)")
    except Exception:
        total = spark.sql(f"SELECT COUNT(*) AS n FROM {fq}").collect()[0]["n"]
        return (
            spark.table(fq)
            .sample(withReplacement=False, fraction=min(1.0, n / max(1, total)), seed=CFG["seed"])
            .limit(n)
        )


def _build_oos_expr(cs: str, safe: str, domain: str) -> str:
    # Build a SQL CASE expression counting out-of-set values for a domain.
    def in_clause(vals):
        return ", ".join(f"'{v}'" for v in vals)

    if domain == "admin_gender":
        canon = in_clause(ADMIN_GENDER_CODES)
        return (f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
                f"AND UPPER(TRIM({cs})) NOT IN ({canon}) "
                f"THEN 1 END) AS `__oos__{safe}`")
    elif domain in ("race", "ethnicity"):
        canon_r = in_clause(OMB_RACE_1997_CODES | OMB_RACE_1997_LABELS)
        canon_e = in_clause(OMB_ETHNICITY_1997_CODES | OMB_ETHNICITY_LABELS)
        canon   = canon_r if domain == "race" else canon_e
        return (f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
                f"AND UPPER(TRIM({cs})) NOT IN ({canon}) "
                f"THEN 1 END) AS `__oos__{safe}`")
    elif domain == "country":
        canon = in_clause(ISO3166_ALPHA2 | {c.lower() for c in ISO3166_ALPHA2})
        return (f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
                f"AND UPPER(TRIM({cs})) NOT IN ({in_clause(ISO3166_ALPHA2)}) "
                f"THEN 1 END) AS `__oos__{safe}`")
    elif domain == "language":
        canon = in_clause(BCP47_COMMON | {c.lower() for c in BCP47_COMMON})
        return (f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
                f"AND LOWER(TRIM({cs})) NOT IN ({in_clause({c.lower() for c in BCP47_COMMON})}) "
                f"THEN 1 END) AS `__oos__{safe}`")
    elif domain == "currency":
        canon = in_clause(ISO4217_CODES)
        return (f"COUNT(CASE WHEN {cs} IS NOT NULL AND TRIM({cs}) != '' "
                f"AND UPPER(TRIM({cs})) NOT IN ({canon}) "
                f"THEN 1 END) AS `__oos__{safe}`")
    else:
        # For clinical codes: no fixed set to check against -- OOS = 0
        return f"CAST(0 AS BIGINT) AS `__oos__{safe}`"


def _build_label_expr(cs: str, safe: str, domain: str) -> str:
    # Count values that look like full labels rather than codes (for BARE_LABEL detection).
    if domain == "admin_gender":
        labels = ", ".join(f"'{l}'" for l in ADMIN_GENDER_LABELS)
        return (f"COUNT(CASE WHEN {cs} IS NOT NULL AND "
                f"UPPER(TRIM({cs})) IN ({labels}) "
                f"THEN 1 END) AS `__label__{safe}`")
    elif domain == "race":
        labels = ", ".join(f"'{l}'" for l in OMB_RACE_1997_LABELS)
        return (f"COUNT(CASE WHEN {cs} IS NOT NULL AND "
                f"UPPER(TRIM({cs})) IN ({labels}) "
                f"THEN 1 END) AS `__label__{safe}`")
    elif domain == "ethnicity":
        labels = ", ".join(f"'{l}'" for l in OMB_ETHNICITY_LABELS)
        return (f"COUNT(CASE WHEN {cs} IS NOT NULL AND "
                f"UPPER(TRIM({cs})) IN ({labels}) "
                f"THEN 1 END) AS `__label__{safe}`")
    else:
        # For other domains: detect long values (avg_length > 10 suggests labels not codes)
        return (f"COUNT(CASE WHEN {cs} IS NOT NULL AND "
                f"LENGTH(TRIM({cs})) > 10 "
                f"THEN 1 END) AS `__label__{safe}`")


def profile_table(tbl: str, code_cols: dict) -> dict:
    # ONE SQL per table for all aggregate stats.
    fq    = f"`{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`"
    exprs = ["COUNT(*) AS __total__"]

    for col, info in code_cols.items():
        cs   = f"`{_esc(col)}`"
        safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
        exprs += [
            f"COUNT(*) - COUNT({cs}) AS `__null__{safe}`",
            f"APPROX_COUNT_DISTINCT({cs}) AS `__acd__{safe}`",
            _build_oos_expr(cs, safe, info["domain"]),
            _build_label_expr(cs, safe, info["domain"]),
        ]

    try:
        row   = spark.sql(f"SELECT {', '.join(exprs)} FROM {fq}").collect()[0].asDict()
        total = row["__total__"] or 0
        for col, info in code_cols.items():
            safe = col.replace("`","").replace(" ","_").replace(".","_")[:60]
            info.update({
                "total":      total,
                "null_count": int(row.get(f"__null__{safe}", 0) or 0),
                "acd":        int(row.get(f"__acd__{safe}",  0) or 0),
                "oos_count":  int(row.get(f"__oos__{safe}",  0) or 0),
                "label_count":int(row.get(f"__label__{safe}",0) or 0),
            })
    except Exception as e:
        print(f"  [WARN] Profile failed for {tbl}: {e}")

    return code_cols


def collect_distinct(tbl: str, col: str, limit: int = 50) -> Dict[str, int]:
    # Collect distinct values with counts for a specific column.
    cs = f"`{_esc(col)}`"
    try:
        rows = spark.sql(f"""
            SELECT TRIM({cs}) AS val, COUNT(*) AS cnt
            FROM `{_esc(cat)}`.`{_esc(sch)}`.`{_esc(tbl)}`
            WHERE {cs} IS NOT NULL AND TRIM({cs}) != ''
            GROUP BY TRIM({cs}) ORDER BY cnt DESC LIMIT {limit}
        """).collect()
        return {r["val"]: int(r["cnt"]) for r in rows}
    except Exception:
        return {}


# Run profiler
table_samples: Dict[str, DataFrame] = {}

for tbl, code_cols in table_code_groups.items():
    sample_df = get_sample(tbl)
    table_samples[tbl] = sample_df

    profile_table(tbl, code_cols)

    # Collect distinct values per column (needed for rule engine and AI)
    for col, info in code_cols.items():
        info["distinct_vals"] = collect_distinct(tbl, col)
        info["sample_vals"]   = list(info["distinct_vals"].keys())[:10]


total_code_cols = sum(len(v) for v in table_code_groups.values())
print(f"Profiler done. {total_code_cols} reference code column(s) across "
      f"{len(table_code_groups)} table(s).")
for tbl, cols in table_code_groups.items():
    print(f"  {tbl}: " + ", ".join(
        f"{col}({info['domain']},oos={info.get('oos_count',0)})"
        for col, info in cols.items()
    ))


In [0]:
# ---------------------------------------------------------------------------
# AI CLASSIFIER -- chunked ai_query() calls.
#
# Three responsibilities:
# 1. DOMAIN CONFIRMATION + REFINEMENT: verify/correct the heuristic domain;
#    distinguish admin_gender vs birth_sex vs gender_identity.
# 2. FREE-TEXT vs CODE detection: does this column store descriptions or codes?
#    (catches "Male"/"Female" vs "M"/"F" when heuristic can't tell from name)
# 3. CONFLATION DETECTION: does a single column appear to mix multiple
#    sex/gender concepts? (admin gender + birth sex in one column)
# ---------------------------------------------------------------------------

BATCH_SIZE = 20

def _ai_query(prompt: str) -> str:
    safe = prompt.replace("'", "''")
    raw  = spark.sql(
        f"SELECT ai_query('{CFG['ai_model']}', '{safe}', failOnError => false) AS r"
    ).collect()[0]["r"]
    if hasattr(raw, "errorStatus") and raw.errorStatus:
        raise ValueError(raw.errorStatus)
    return raw.result if hasattr(raw, "result") else str(raw)

def _chunked(items: list, size: int = BATCH_SIZE):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def classify_table(tbl: str) -> None:
    code_cols = table_code_groups.get(tbl, {})
    if not code_cols:
        return

    if not CFG["enable_ai"]:
        for col, info in code_cols.items():
            info["ai_domain"]     = info["heuristic_domain"]
            info["ai_confirmed"]  = True
            info["ai_confidence"] = "low"
            info["is_free_text"]  = info.get("label_count", 0) > 0
            info["is_conflated"]  = False
        return

    col_list = list(code_cols.keys())
    for chunk in _chunked(col_list):
        payload = json.dumps([{
            "col":            col,
            "table":          tbl,
            "heuristic_domain": code_cols[col]["heuristic_domain"],
            "top_values":     code_cols[col].get("sample_vals", [])[:15],
            "distinct_count": code_cols[col].get("acd", 0),
            "total_rows":     code_cols[col].get("total", 0),
            "has_companion_system":  code_cols[col].get("companion_system") is not None,
            "has_companion_version": code_cols[col].get("companion_version") is not None,
        } for col in chunk if col in code_cols], default=str)

        prompt = (
            "Healthcare data governance expert. Reply ONLY with a JSON array -- no prose, no markdown. "
            f"Table: {tbl}. For each reference code column determine: "
            "(1) domain: one of admin_gender, birth_sex, gender_identity, sexual_orientation, "
            "race, ethnicity, country, language, currency, ucum_unit, icd, loinc, snomed, "
            "rxnorm, cvx, ndc, npi, other_code, not_a_code. "
            "(2) is_free_text: true if column stores full text descriptions (Male/Female, White/Black) "
            "rather than standard codes (M/F, 2106-3). False if values look like codes. "
            "(3) is_conflated: true only for gender/sex columns where the values appear to "
            "mix multiple concepts (e.g. storing birth sex and gender identity together). "
            "(4) confidence: high/medium/low. "
            f"Columns: {payload}. "
            'Return: [{"col":"<n>","domain":"<d>","is_free_text":true|false,'
            '"is_conflated":false,"confidence":"high|medium|low"}]'
        )
        try:
            for item in json.loads(_ai_query(prompt)):
                col = item.get("col", "")
                if col not in code_cols:
                    continue
                code_cols[col].update({
                    "ai_domain":     item.get("domain", code_cols[col]["heuristic_domain"]),
                    "ai_confirmed":  item.get("domain", "not_a_code") != "not_a_code",
                    "ai_confidence": item.get("confidence", "low"),
                    "is_free_text":  item.get("is_free_text", False),
                    "is_conflated":  item.get("is_conflated", False),
                })
        except Exception as e:
            print(f"  [WARN] AI classification failed for {tbl}: {e}")
            for col in chunk:
                if col in code_cols:
                    code_cols[col].update({
                        "ai_domain":     code_cols[col]["heuristic_domain"],
                        "ai_confirmed":  True,
                        "ai_confidence": "low",
                        "is_free_text":  code_cols[col].get("label_count", 0) > 0,
                        "is_conflated":  False,
                        "ai_error":      str(e),
                    })

    # Remove columns AI confirmed are not reference codes (at high confidence only)
    for col in list(table_code_groups[tbl].keys()):
        info = table_code_groups[tbl][col]
        if not info.get("ai_confirmed", True) and info.get("ai_confidence") == "high":
            del table_code_groups[tbl][col]


for tbl in list(table_code_groups.keys()):
    classify_table(tbl)
    remaining = len(table_code_groups[tbl])
    domains = {}
    for info in table_code_groups[tbl].values():
        d = info.get("ai_domain", "?")
        domains[d] = domains.get(d, 0) + 1
    dom_str = ", ".join(f"{d}:{n}" for d, n in sorted(domains.items()))
    print(f"  ok {tbl}: {remaining} code column(s) [{dom_str}]")

total_confirmed = sum(len(v) for v in table_code_groups.values())
print(f"\nAI classification done. {total_confirmed} confirmed reference code column(s).")


In [0]:
# ---------------------------------------------------------------------------
# RULE ENGINE -- one _check_code_col() per column; table-level checks
# for gender field completeness and race/ethnicity rollup.
# ---------------------------------------------------------------------------

def _esc(name: str) -> str:
    return name.replace("`", "``")


def _finding(tbl, col, tag, count, total, samples, action,
             hint=None, confidence="high", std_options=None,
             auto_decided_action=None, domain="other_code") -> dict:
    pct     = round(count / total * 100, 4) if total else 0.0
    options = std_options if std_options is not None else STANDARDIZATION_RULES.get(tag, [])
    decided = auto_decided_action if auto_decided_action is not None else None
    return {
        "run_id":                   RUN_ID,
        "run_ts":                   RUN_TS,
        "catalog":                  cat,
        "schema":                   sch,
        "table_name":               tbl,
        "column_name":              col,
        "domain":                   domain,
        "layer":                    CFG["layer"],
        "rule_ref":                 TAGS.get(tag, "§?"),
        "classification":           tag,
        "affected_count":           int(count),
        "affected_pct":             float(pct),
        "total_rows":               int(total),
        "sample_values":            json.dumps(samples, default=str),
        "recommended_action":       action,
        "dictionary_strategy_hint": hint,
        "standardization_rule":     json.dumps(options, ensure_ascii=False),
        "confidence":               confidence,
        "needs_steward_review":     decided is None,
        "decided_action":           decided,
        "decided_by":               None,
    }


def _check_code_col(tbl: str, col: str, info: dict) -> list:
    total      = info.get("total", 0)
    null_count = info.get("null_count", 0)
    non_null   = total - null_count
    domain     = info.get("ai_domain", info.get("heuristic_domain", "other_code"))
    confidence = info.get("ai_confidence", "low")
    oos_count  = info.get("oos_count", 0)
    label_count= info.get("label_count", 0)
    is_free_text= info.get("is_free_text", False)
    is_conflated= info.get("is_conflated", False)
    has_system  = info.get("companion_system") is not None
    has_version = info.get("companion_version") is not None
    has_rollup  = info.get("companion_rollup") is not None
    sample_vals = info.get("sample_vals", [])
    distinct_vals= info.get("distinct_vals", {})
    findings    = []

    # ── §Validation: HIGH_NULL_RATE ───────────────────────────────────────────
    if total > 0 and null_count / total * 100 > CFG["null_threshold"]:
        findings.append(_finding(tbl, col, "HIGH_NULL_RATE",
            null_count, total, [],
            f"{null_count/total*100:.1f}% NULL in reference code column '{col}' (domain: {domain}). "
            "May indicate upstream mapping failure.",
            confidence="medium", domain=domain,
        ))

    # ── §Standards catalog: MISSING_CODE_SYSTEM ──────────────────────────────
    # Clinical and coded columns should always have a companion code_system
    if not has_system and domain not in ("ucum_unit", "not_a_code", "other_code"):
        findings.append(_finding(tbl, col, "MISSING_CODE_SYSTEM",
            0, total, sample_vals[:3],
            f"Coded column '{col}' (domain: {domain}) has no companion code_system column. "
            "Store code + code system + version per §2. "
            "Add <col>_code_system column with documented default system.",
            confidence="high", domain=domain,
        ))

    # ── §Standards catalog: MISSING_VERSION ──────────────────────────────────
    # Race/ethnicity columns are excluded here: they get VERSION_TRANSITION_RISK
    # below which is more specific (includes OMB 2024 context and transition timeline).
    # Firing both for the same column produces duplicate steward work items for the
    # same gap. Clinical codes (ICD, LOINC, SNOMED, RxNorm, CVX) still fire here.
    if not has_version and domain in ("icd", "loinc", "snomed", "rxnorm", "cvx"):
        findings.append(_finding(tbl, col, "MISSING_VERSION",
            0, total, sample_vals[:3],
            f"Coded column '{col}' (domain: {domain}) has no companion version column. "
            "Version transitions must be tracked (§2). "
            "Add <col>_std_version column with documented default "
            "(e.g. ICD-10-CM-2024, LOINC-2.76, SNOMED-CT-2024-03).",
            confidence="high", domain=domain,
        ))

    # ── §General handling: FREE_TEXT_WHERE_CODE_EXISTS ────────────────────────
    label_pct = label_count / non_null * 100 if non_null else 0
    if (is_free_text or label_pct > CFG["label_threshold"]) and non_null > 0:
        label_samples = [v for v in distinct_vals if len(v) > 5][:5]
        tag = "UNCODED_FREE_TEXT_CERTIFIED" if CFG["layer"] in ("Gold",) else "FREE_TEXT_WHERE_CODE_EXISTS"
        findings.append(_finding(tbl, col, tag,
            label_count, total, label_samples,
            f"Column '{col}' (domain: {domain}) stores full labels/descriptions "
            f"({label_pct:.1f}% of values) rather than standard codes. "
            "Replace with coded values in Silver ETL (§2, §6). "
            + ("Certification blocker in Gold layer." if tag == "UNCODED_FREE_TEXT_CERTIFIED" else ""),
            confidence="high" if is_free_text else "medium",
            domain=domain,
        ))

    # ── §Validation: OUT_OF_SET_VALUE + HIGH_OOS_RATE ─────────────────────────
    if oos_count > 0 and domain in (
        "admin_gender","race","ethnicity","country","language","currency"
    ):
        oos_samples = [v for v in distinct_vals if _is_oos(v, domain)][:8]
        findings.append(_finding(tbl, col, "OUT_OF_SET_VALUE",
            oos_count, total, oos_samples,
            f"{oos_count:,} value(s) in '{col}' are not in the governing code set for {domain}. "
            "DQ rules must fail or flag rows with out-of-set values (§6). "
            "NULL or quarantine; do not pass to Gold without mapping.",
            confidence="high", domain=domain,
        ))
        if non_null > 0 and oos_count / non_null * 100 > CFG["oos_threshold"]:
            findings.append(_finding(tbl, col, "HIGH_OOS_RATE",
                oos_count, total, oos_samples[:3],
                f"{oos_count/non_null*100:.1f}% of non-null values in '{col}' are out-of-set. "
                "Certified demographic/coded dimensions with non-standard codes "
                "are certification blockers (§Enforcement).",
                confidence="high",
                auto_decided_action="escalate_and_quarantine",
                domain=domain,
            ))

    # ── §General handling: BARE_LABEL_STORED ─────────────────────────────────
    if label_count > 0 and not is_free_text and domain in ("admin_gender","race","ethnicity"):
        label_samples = [v for v in distinct_vals if _is_label(v, domain)][:5]
        if label_samples:
            findings.append(_finding(tbl, col, "BARE_LABEL_STORED",
                label_count, total, label_samples,
                f"{label_count:,} value(s) in '{col}' appear to be full labels "
                f"(e.g. {label_samples[:2]}) rather than canonical codes. "
                "Map to canonical codes in Silver ETL (§2). "
                "Register label → code mappings in alias table.",
                confidence="high", domain=domain,
            ))

    # ── §General handling: VERSION_NOT_TRACKED ────────────────────────────────
    if domain in ("race", "ethnicity") and not has_version:
        findings.append(_finding(tbl, col, "VERSION_TRANSITION_RISK",
            0, total, [],
            f"Race/ethnicity column '{col}' has no OMB version tracking. "
            "OMB 2024 finalized 2024-03-28; federal compliance by 2029-03-28 (§4). "
            "Add race_std_version column; model changes must be additive for MENA category.",
            confidence="high", domain=domain,
        ))

    # ── §Sex and gender: INVALID_ADMIN_GENDER ────────────────────────────────
    if domain == "admin_gender" and oos_count > 0:
        inv_samples = [v for v in distinct_vals
                       if v.upper().strip() not in ADMIN_GENDER_CODES
                       and v.upper().strip() not in ADMIN_GENDER_LABELS][:5]
        if inv_samples or oos_count > 0:
            findings.append(_finding(tbl, col, "INVALID_ADMIN_GENDER",
                oos_count, total, (inv_samples or list(distinct_vals.keys()))[:5],
                f"{oos_count:,} value(s) in '{col}' are not valid FHIR AdministrativeGender codes. "
                "Valid codes: M, F, O, U, N (§3). "
                "Map to canonical codes; unmapped → U or quarantine per steward rule.",
                confidence="high", domain=domain,
            ))

    # ── §Sex and gender: GENDER_FIELDS_CONFLATED ──────────────────────────────
    if is_conflated:
        findings.append(_finding(tbl, col, "GENDER_FIELDS_CONFLATED",
            0, total, sample_vals[:3],
            f"Column '{col}' appears to conflate multiple sex/gender concepts. "
            "USCDI requires separate columns: admin_gender, birth_sex, gender_identity (§3). "
            "Do not collapse into a single column.",
            confidence=confidence, domain=domain,
        ))

    # ── §Sex and gender: SOGI_PHI_RISK ───────────────────────────────────────
    if domain in ("gender_identity", "sexual_orientation"):
        findings.append(_finding(tbl, col, "SOGI_PHI_RISK",
            non_null, total, [],
            f"Column '{col}' stores {domain.replace('_',' ')} data -- PHI-sensitive (§3). "
            "Apply PHI storage and handling; minimum-necessary access enforcement. "
            "Verify column-level security in Unity Catalog.",
            confidence="high",
            auto_decided_action="verify_phi_controls",
            domain=domain,
        ))

    # ── §Race and ethnicity: MISSING_ROLLUP_LOOKUP ────────────────────────────
    if domain in ("race", "ethnicity") and not has_rollup:
        findings.append(_finding(tbl, col, "MISSING_ROLLUP_LOOKUP",
            0, total, [],
            f"Race/ethnicity column '{col}' has no companion rollup/OMB-category column detected. "
            "Every detailed CDCREC value must map to exactly one OMB minimum category (§6). "
            "Discarding source granularity without rollup is a policy violation (§Enforcement). "
            "Create CDCREC → OMB rollup hierarchy lookup table.",
            confidence="high", domain=domain,
        ))

    # ── §Race and ethnicity: INVALID_OMB_CATEGORY ────────────────────────────
    if domain == "race" and oos_count > 0:
        inv_samples = [v for v in distinct_vals
                       if v.upper().strip() not in (OMB_RACE_1997_CODES | OMB_RACE_1997_LABELS
                                                    | OMB_RACE_2024_ADDITIONS)][:5]
        if inv_samples:
            findings.append(_finding(tbl, col, "INVALID_OMB_CATEGORY",
                len(inv_samples), total, inv_samples,
                f"Race column '{col}' contains values not in OMB SPD 15 categories. "
                "Map to valid OMB categories via CDCREC rollup (§4). "
                "Keep most granular source value; roll up via CDCREC hierarchy.",
                confidence="medium", domain=domain,
            ))

    return findings


def _is_oos(val: str, domain: str) -> bool:
    # Check if a value is out-of-set for its domain.
    v = val.upper().strip()
    if domain == "admin_gender":
        return v not in ADMIN_GENDER_CODES and v not in ADMIN_GENDER_LABELS
    if domain == "race":
        return v not in OMB_RACE_1997_CODES and v not in OMB_RACE_1997_LABELS
    if domain == "ethnicity":
        return v not in OMB_ETHNICITY_1997_CODES and v not in OMB_ETHNICITY_LABELS
    if domain == "country":
        return v not in ISO3166_ALPHA2
    if domain == "language":
        return val.lower().strip() not in {c.lower() for c in BCP47_COMMON}
    if domain == "currency":
        return v not in ISO4217_CODES
    return False


def _is_label(val: str, domain: str) -> bool:
    # Check if a value looks like a full label rather than a short code.
    v = val.upper().strip()
    if domain == "admin_gender":
        return v in ADMIN_GENDER_LABELS
    if domain == "race":
        return v in OMB_RACE_1997_LABELS
    if domain == "ethnicity":
        return v in OMB_ETHNICITY_LABELS
    return False


# ── Table-level checks ────────────────────────────────────────────────────────
def _check_table_level(tbl: str, code_cols: dict) -> list:
    # Checks that require looking across all code columns in a table.
    findings = []
    domains  = {info.get("ai_domain", "?") for info in code_cols.values()}
    total    = next((info.get("total", 0) for info in code_cols.values()), 0)

    # §Sex and gender: MISSING_BIRTH_SEX
    # Table has admin_gender but no birth_sex column
    has_admin_gender = any(
        info.get("ai_domain") == "admin_gender"
        for info in code_cols.values()
    )
    has_birth_sex = any(
        info.get("ai_domain") == "birth_sex"
        for info in code_cols.values()
    )
    # Only fire in Silver/Gold -- Bronze may have source-faithful data
    if has_admin_gender and not has_birth_sex and CFG["layer"] in ("Silver", "Gold"):
        findings.append(_finding(tbl, "-- table --", "MISSING_BIRTH_SEX",
            0, total, [],
            f"Table has administrative gender column but no sex_assigned_at_birth / birth_sex column. "
            "USCDI requires sex assigned at birth as distinct from administrative gender (§3). "
            "Add birth_sex column and document the distinction.",
            confidence="medium", domain="birth_sex",
        ))

    # §Race and ethnicity: MENA_NOT_MODELED (informational, forward-looking)
    has_race = "race" in domains
    if has_race and CFG["layer"] in ("Silver", "Gold"):
        # Check if any column name or value suggests OMB 2024 MENA awareness
        mena_aware = any(
            "mena" in col.lower() or "2024" in col.lower()
            for col in code_cols
        )
        if not mena_aware:
            findings.append(_finding(tbl, "-- table --", "MENA_NOT_MODELED",
                0, total, [],
                "Race model appears to use OMB 1997 categories only with no MENA provision. "
                "OMB 2024 (finalized 2024-03-28) adds Middle Eastern or North African as a 7th category. "
                "Federal compliance required by 2029-03-28. "
                "Plan additive schema extension -- do not hard-code 5+2 shape (§4).",
                confidence="low", domain="race",
            ))

    return findings


# ── Main loop ─────────────────────────────────────────────────────────────────
all_findings: List[dict] = []

for tbl, code_cols in table_code_groups.items():
    tbl_count = 0

    # Column-level checks
    for col, info in code_cols.items():
        col_findings = _check_code_col(tbl, col, info)
        all_findings.extend(col_findings)
        tbl_count += len(col_findings)

    # Table-level checks
    tbl_findings = _check_table_level(tbl, code_cols)
    all_findings.extend(tbl_findings)
    tbl_count += len(tbl_findings)

    print(f"  ok {tbl}: {tbl_count} finding(s)")

print(f"\nRule engine done. {len(all_findings)} total finding(s).")


In [0]:
from pyspark.sql.types import (StructType, StructField, StringType,
                                LongType, DoubleType, BooleanType)

SCHEMA = StructType([
    StructField("run_id",                   StringType(),  True),
    StructField("run_ts",                   StringType(),  True),
    StructField("catalog",                  StringType(),  True),
    StructField("schema",                   StringType(),  True),
    StructField("table_name",               StringType(),  True),
    StructField("column_name",              StringType(),  True),
    StructField("domain",                   StringType(),  True),
    StructField("layer",                    StringType(),  True),
    StructField("rule_ref",                 StringType(),  True),
    StructField("classification",           StringType(),  True),
    StructField("affected_count",           LongType(),    True),
    StructField("affected_pct",             DoubleType(),  True),
    StructField("total_rows",               LongType(),    True),
    StructField("sample_values",            StringType(),  True),
    StructField("recommended_action",       StringType(),  True),
    StructField("dictionary_strategy_hint", StringType(),  True),
    StructField("standardization_rule",     StringType(),  True),
    StructField("confidence",               StringType(),  True),
    StructField("needs_steward_review",     BooleanType(), True),
    StructField("decided_action",           StringType(),  True),
    StructField("decided_by",               StringType(),  True),
])

out_fq  = f"`{CFG['out_catalog']}`.`{CFG['out_schema']}`.`{CFG['out_table']}`"
out_tbl = f"{CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}"

findings_df = spark.createDataFrame(all_findings or [], schema=SCHEMA)

if all_findings:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CFG['out_catalog']}`.`{CFG['out_schema']}`")
    findings_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(out_tbl)
    print(f"ok  {len(all_findings):,} finding(s) written to {out_fq}")
else:
    print("No findings generated.")

print(f"Run ID: {RUN_ID}")


In [0]:
BLOCKING = {
    "OUT_OF_SET_VALUE", "HIGH_OOS_RATE", "INVALID_ADMIN_GENDER",
    "UNCODED_FREE_TEXT_CERTIFIED", "MISSING_CODE_SYSTEM",
    "GENDER_FIELDS_CONFLATED", "INVALID_OMB_CATEGORY",
}

if not all_findings:
    print("No reference code findings. Run complete.")
else:
    fdf = findings_df

    # Section 1 -- Blocking
    block_df = fdf.filter(F.col("classification").isin(BLOCKING)) \
                  .orderBy(F.col("affected_pct").desc())
    n_block = block_df.count()
    print("=" * 70)
    print(f"  SECTION 1 -- BLOCKING FINDINGS: {n_block}")
    print("=" * 70)
    if n_block:
        display(block_df.select(
            "table_name","column_name","domain","layer","classification",
            "rule_ref","affected_count","affected_pct",
            "sample_values","recommended_action","decided_action","decided_by"
        ))
    else:
        print("  None.")

    # Section 2 -- By classification
    print("\n" + "-" * 70)
    print("SECTION 2 -- Findings by classification")
    print("-" * 70)
    display(
        fdf.groupBy("classification","rule_ref","domain")
           .agg(
               F.count("*").alias("findings"),
               F.countDistinct("table_name").alias("tables"),
               F.sum("affected_count").alias("total_affected"),
           ).orderBy(F.col("findings").desc())
    )

    # Section 3 -- Standards catalog gaps
    catalog_df = fdf.filter(F.col("classification").isin(
        "MISSING_CODE_SYSTEM","MISSING_VERSION","WRONG_CODE_SYSTEM","VERSION_NOT_TRACKED"
    ))
    n_catalog = catalog_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 3 -- Standards catalog gaps ({n_catalog})")
    print("  Code + system + version required per §2")
    print("-" * 70)
    if n_catalog:
        display(catalog_df.select(
            "table_name","column_name","domain","classification",
            "recommended_action","decided_action","decided_by"
        ).orderBy("table_name","domain"))

    # Section 4 -- Sex/gender
    gender_df = fdf.filter(F.col("domain").isin(
        "admin_gender","birth_sex","gender_identity","sexual_orientation"
    ))
    n_gender = gender_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 4 -- Sex and gender findings ({n_gender})")
    print("  USCDI requires separate columns per concept; PHI-sensitive")
    print("-" * 70)
    if n_gender:
        display(gender_df.select(
            "table_name","column_name","domain","classification",
            "affected_count","sample_values","recommended_action","decided_action","decided_by"
        ).orderBy("table_name","domain","classification"))

    # Section 5 -- Race and ethnicity
    race_df = fdf.filter(F.col("domain").isin("race","ethnicity"))
    n_race = race_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 5 -- Race and ethnicity findings ({n_race})")
    print("  OMB SPD 15 + CDCREC; OMB 2024 transition planning required")
    print("-" * 70)
    if n_race:
        display(race_df.select(
            "table_name","column_name","domain","classification",
            "affected_count","sample_values","recommended_action","decided_action","decided_by"
        ).orderBy("table_name","classification"))

    # Section 6 -- Clinical terminologies
    clinical_df = fdf.filter(F.col("domain").isin(
        "icd","loinc","snomed","rxnorm","cvx","ndc","npi","other_code"
    ))
    n_clinical = clinical_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 6 -- Clinical terminology findings ({n_clinical})")
    print("  ICD/LOINC/SNOMED/RxNorm/CVX/NDC/NPI: code + system + version required")
    print("-" * 70)
    if n_clinical:
        display(clinical_df.select(
            "table_name","column_name","domain","classification",
            "recommended_action","decided_action","decided_by"
        ).orderBy("table_name","domain"))

    # Section 7 -- Work queue
    work_df = fdf.filter(F.col("decided_action").isNotNull())
    n_work  = work_df.count()
    print("\n" + "-" * 70)
    print(f"SECTION 7 -- Engineer work queue ({n_work} decided)")
    print("-" * 70)
    if n_work:
        display(work_df.select(
            "table_name","column_name","domain","classification",
            "decided_action","decided_by","standardization_rule"
        ).orderBy("table_name","domain"))
    else:
        print("  No decisions recorded yet.")
        print(f"  Query: SELECT * FROM {CFG['out_catalog']}.{CFG['out_schema']}.{CFG['out_table']}")
        print(f"  WHERE run_id = '{RUN_ID}' AND decided_action IS NULL")

    print("\n" + "=" * 70)
    print(f"  Run: {RUN_ID}")
    print(f"  Scope: {cat}.{sch}  |  Layer: {CFG['layer']}")
    print(f"  Code columns confirmed: {total_confirmed}")
    print(f"  Findings: {len(all_findings):,}  |  Blocking: {n_block}  |  Catalog gaps: {n_catalog}  |  Gender: {n_gender}  |  Race/eth: {n_race}  |  Clinical: {n_clinical}")
    print("=" * 70)
    print("\nDetection-only. No source data was modified.")
    print("Every finding requires data steward review before any action.")
